<a href="https://www.kaggle.com/code/helgabuchhausel/fatigue-dataset?scriptVersionId=280395199" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:

import numpy as np 
import pandas as pd 
import os
import numpy as np
from PIL import Image 
import glob
from collections import Counter
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch import torch
from torchvision import models
from torch.utils.data import random_split


for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/fatigue-dataset/Data/Fatigue/0664.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0733.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0106.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0375.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/1075.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0285.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0591.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0799.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0074.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/1031.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0077.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0498.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0610.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0617.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/1024.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0426.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0989.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0235.jpg
/kaggle/input/fatigue-dataset/Data/Fatigue/0273.jpg
/kaggle/inpu

In [23]:
path = '/kaggle/input/fatigue-dataset'
fatigue_dir = os.path.join(path, 'Data/Fatigue')
no_fatigue_dir = os.path.join(path, 'Data/NonFatigue')


fatigue_files = glob.glob(os.path.join(fatigue_dir, '*.[pj][pn]g'))
no_fatigue_files = glob.glob(os.path.join(no_fatigue_dir, '*.[pj][pn]g'))

N_fatigue = len(fatigue_files)
N_no_fatigue = len(no_fatigue_files)
N_total = N_fatigue + N_no_fatigue


print(f"Fatigue Samples: {N_fatigue}")
print(f"No-Fatigue Samples: {N_no_fatigue}")
print(f"Total Samples: {N_total}")

Fatigue Samples: 1100
No-Fatigue Samples: 1100
Total Samples: 2200


In [24]:
min_count = min(N_fatigue, N_no_fatigue)
max_count = max(N_fatigue, N_no_fatigue)

if max_count == 0:
    imbalance_ratio = 0.0
else:
    imbalance_ratio = min_count / max_count

if (imbalance_ratio >= 0.95):
    print("Highly balanced")
elif (imbalance_ratio > 0.5):
    print("Moderate imbalance")
else:
    print("Severe imbalance")

print(f"Minority Class Count: {min_count}")
print(f"Majority Class Count: {max_count}")
print(f"Imbalance Ratio: {imbalance_ratio:.4f}")

Highly balanced
Minority Class Count: 1100
Majority Class Count: 1100
Imbalance Ratio: 1.0000


In [25]:
all_files = fatigue_files + no_fatigue_files

resolutions = []
channels = []
problem_files = 0

for file_path in all_files:
    try:
        with Image.open(file_path) as img:
            resolutions.append(img.size)
            channels.append(len(img.getbands())) 
    except Exception:
        problem_files += 1 
        
print(f"Files that could not be opened: {problem_files}")

Files that could not be opened: 0


In [26]:
res_array = np.array(resolutions)

widths = res_array[:,0]
heights = res_array[:,1]

print(f"Mean Resolution (W x H): {widths.mean():.0f} x {heights.mean():.0f} pixels")
print(f"Std Dev (W x H): {widths.std():.0f} x {heights.std():.0f} pixels")

Mean Resolution (W x H): 242 x 242 pixels
Std Dev (W x H): 59 x 59 pixels


In [27]:
channel_counts = Counter(channels)
print(f"Channel Counts: {channel_counts}")
print(f"Total files analyzed: {len(all_files) - problem_files}")

Channel Counts: Counter({3: 2200})
Total files analyzed: 2200


In [28]:
mean_w = widths.mean()
std_dev_w = widths.std()
mean_h = heights.mean()
std_dev_h = heights.std()

## standar cnn input target
target_size = 224.0

def calculate_variance_ratio(mean, std_dev, target):
    if target == 0: return 0.0
    return std_dev / target

# Calculate the severity ratio for width and height
R_var_W = calculate_variance_ratio(mean_w, std_dev_w, target_size)
R_var_H = calculate_variance_ratio(mean_h, std_dev_h, target_size)

print(f"Target Input Size: {target_size:.0f} x {target_size:.0f} pixels")
print(f"Width Variance Ratio (R_var_W): {R_var_W:.3f}")
print(f"Height Variance Ratio (R_var_H): {R_var_H:.3f}")

# Critical Threshold: If the std dev is more than 30% of the target size, it's severe.
CRITICAL_THRESHOLD = 0.30


Target Input Size: 224 x 224 pixels
Width Variance Ratio (R_var_W): 0.264
Height Variance Ratio (R_var_H): 0.264


In [34]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
TARGET_SIZE = 224

data_transforms = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)), 
    
    
    transforms.ToTensor(), 
    
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) 
])

print("\n--- Final Pre-processing Pipeline Implemented ---")
print("The data_transforms object now contains the mandatory steps for Resizing and Standardization.")


--- Final Pre-processing Pipeline Implemented ---
The data_transforms object now contains the mandatory steps for Resizing and Standardization.


In [35]:
try:
    sample_file_path = glob.glob(os.path.join(fatigue_dir, '*.[pj][pn]g'))[0]
except IndexError:
    print("WARNING: Could not access local files. Using a placeholder path for code execution example.")

transform_pipeline = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) 
])

def process_and_test_image(file_path, transform):
    print(f"\nProcessing file: {os.path.basename(file_path)}")
    
    try:
        img = Image.open(file_path)
        print(f"Initial size: {img.size}, Initial channels: {len(img.getbands())}")

        
        img_rgb = img.convert('RGB')
        print(f"After 'RGB' conversion: Channels: {len(img_rgb.getbands())}")

        tensor_output = transform(img_rgb)
        

        print(f"Final Tensor Shape: {tensor_output.shape}")
        
        if tensor_output.shape[0] == 3:
            print("VERDICT: Success. The output tensor has 3 channels and is standardized.")
        else:
            print("VERDICT: **FAILURE**. Tensor channel count is incorrect.")
            
        print(f"Tensor Min Value: {tensor_output.min():.4f}, Tensor Max Value: {tensor_output.max():.4f}")

    except FileNotFoundError:
        print(f"ERROR: Sample file not found at {file_path}. Cannot test locally.")
    except Exception as e:
        print(f"CRITICAL ERROR during processing: {e}")


process_and_test_image(sample_file_path, transform_pipeline)


Processing file: 0664.jpg
Initial size: (244, 244), Initial channels: 3
After 'RGB' conversion: Channels: 3
Final Tensor Shape: torch.Size([3, 224, 224])
VERDICT: Success. The output tensor has 3 channels and is standardized.
Tensor Min Value: -2.1179, Tensor Max Value: 2.6400


In [37]:
class FatigueDataset(Dataset):
    """A custom Dataset class for the Fatigue/NonFatigue images."""
    
    # 1. Initialization (Run once)
    def __init__(self, root_dir, transform=None):
        """
        Args:
            root_dir (str): Directory with 'fatigue' and 'no_fatigue' subfolders.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        self.labels = [] # 0 for NonFatigue, 1 for Fatigue

        class_map = {"NonFatigue": 0, "Fatigue": 1}
        
        for class_name, label in class_map.items():
            class_dir = os.path.join(self.root_dir, class_name)
            
            paths = []
            paths.extend(glob.glob(os.path.join(class_dir, '*.jpg')))
            paths.extend(glob.glob(os.path.join(class_dir, '*.jpeg')))
            paths.extend(glob.glob(os.path.join(class_dir, '*.png')))
            
            self.image_paths.extend(paths)
            self.labels.extend([label] * len(paths))
        
        print(f"DEBUG: Indexing found {len(self.image_paths)} total files.")
        if len(self.image_paths) == 0:
            print(f"FATAL: No images found in {root_dir}. Check the path and folder structure.")


    def __len__(self):
        return len(self.image_paths)


    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()

        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        try:
            image = Image.open(img_path)
            image = image.convert('RGB') 

            if self.transform:
                image = self.transform(image)
        
            return image, label
        except Exception as e:
            print(f"Skipping corrupted file at index {idx}: {img_path}. Error: {e}")
            
            new_idx = (idx + 1) % len(self)
            return self.__getitem__(new_idx) 

In [40]:
train_transforms = transforms.Compose([
    transforms.Resize((TARGET_SIZE, TARGET_SIZE)),
    transforms.RandomHorizontalFlip(), # A simple, essential data augmentation step
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

DATA_ROOT = path + "/Data"

In [41]:
full_dataset = FatigueDataset(root_dir=DATA_ROOT, transform=train_transforms)


BATCH_SIZE = 50
NUM_WORKERS = 0

train_loader = DataLoader(
    full_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS 
)

print("\n--- Data Loader Ready ---")
print(f"Total samples indexed: {len(full_dataset)}")
print(f"Batches per epoch: {len(train_loader)}")


DEBUG: Indexing found 2200 total files.

--- Data Loader Ready ---
Total samples indexed: 2200
Batches per epoch: 44


In [42]:
NUM_CLASSES = 2 


model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)


num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, NUM_CLASSES)


DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
print(f"Model sent to device: {DEVICE}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 149MB/s]

Model sent to device: cpu


In [43]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [44]:
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    
    model.train() 
    running_loss = 0.0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        
        images = images.to(device)
        labels = labels.to(device)

 
        optimizer.zero_grad() 


        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        if (batch_idx + 1) % 100 == 0:
            print(f"Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}")

    epoch_loss = running_loss / len(dataloader.dataset)
    print(f"\nEpoch complete. Average Loss: {epoch_loss:.4f}")
    return epoch_loss


In [45]:
NUM_EPOCHS = 100
print(f"\nStarting training for {NUM_EPOCHS} epochs...")

for epoch in range(NUM_EPOCHS):
    print(f"\n--- Epoch {epoch + 1}/{NUM_EPOCHS} ---")
      
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)


Starting training for 100 epochs of slow, painful iteration...

--- Epoch 1/100 ---


KeyboardInterrupt: 